[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/gemma_from_scratch/blob/main/workshop/08_gemma_model_assembly.ipynb)


# 08 - Full Gemma 3 Model Assembly

[← 07 The Transformer Block](07_the_transformer_block.ipynb) | [09 Inference and Sampling →](09_inference_and_sampling.ipynb)


**Estimated Time: 15 minutes**

This is the big moment! We assemble **every component** built in previous notebooks into a complete Gemma 3 decoder-only model. The model runs from token IDs to text generation.

## Learning Objectives:
1. Assemble all components into a complete `Gemma3Model` class.
2. Understand the forward pass: embedding → transformer blocks → output.
3. Build a greedy text generator.
4. See the full architecture in action -- a working 270M-parameter Gemma 3 model.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

# Gemma 3 architecture parameters (~270M)
vocab_size = 256_000  # Gemma 3 tokenizer: 256K vocab
hidden_size = 768
num_layers = 8  # Number of transformer blocks
num_heads = 8
num_kv_heads = 2
num_kv_groups = num_heads // num_kv_heads  # = 4
head_dim = hidden_size // num_heads  # = 96
intermediate_size = 2048
sliding_window = 1024
max_position_embeddings = 512
rope_theta_local = 10_000.0
rope_theta_global = 1_000_000.0 * 8.0  # Rescaled for 128K context
logit_cap = 30.0
hidden_scale = hidden_size**0.5  # = sqrt(768)

Python version: 3.12.13 (main, Mar 10 2026, 18:15:41) [Clang 21.1.4 ]
PyTorch version: 2.5.1


---

## 1. All Components Reused From Previous Notebooks


In [2]:
class Gemma3RMSNorm(nn.Module):
    """Gemma 3-style RMSNorm with Add-One initialization."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        return self._norm(x.float()).type_as(x) * (1.0 + self.weight)

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)


class Gemma3GQA(nn.Module):
    """Grouped Query Attention with QK-Norm and RoPE (Gemma 3)."""

    def __init__(
        self, d_in, n_heads, n_kv_groups, h_dim, is_local=True, sliding_window=None
    ):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups
        self.h_dim = h_dim
        self.group_size = n_heads // n_kv_groups
        self.is_local = is_local
        self.sliding_window = sliding_window

        self.W_q = nn.Linear(d_in, n_heads * h_dim, bias=False)
        self.W_k = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.W_v = nn.Linear(d_in, n_kv_groups * h_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * h_dim, d_in, bias=False)

        # Dual-frequency RoPE: global layers use much higher theta
        self.rope_theta = rope_theta_global if not is_local else rope_theta_local

    def _compute_rope(self, x, offset=0):
        """Compute and apply RoPE to a (B, H, T, D) tensor."""
        B, H, T, D = x.shape
        inv_freq = 1.0 / (
            self.rope_theta ** (torch.arange(0, D, 2, device=x.device).float() / D)
        )
        t = torch.arange(T, device=x.device).float() + offset
        freqs = torch.outer(t, inv_freq)  # (T, D//2)
        cos, sin = freqs.cos(), freqs.sin()  # (T, D//2) — broadcast over B, H

        x_left = x[..., : D // 2]
        x_right = x[..., D // 2 :]
        # Rotation: x'_left = x_left*cos - x_right*sin
        #           x'_right = x_left*sin + x_right*cos
        rotated_left = x_left * cos - x_right * sin
        rotated_right = x_left * sin + x_right * cos
        return torch.cat([rotated_left, rotated_right], dim=-1)

    def _apply_causal_mask(self, scores):
        """Apply causal mask + optional sliding window mask."""
        T_q, T_k = scores.shape[-2], scores.shape[-1]
        # Standard causal mask: upper triangle = future tokens
        causal = torch.triu(
            torch.ones(T_q, T_k, dtype=torch.bool, device=scores.device), diagonal=1
        )
        if self.is_local and self.sliding_window is not None:
            # Also mask keys more than sliding_window steps in the past
            far_past = torch.tril(
                torch.ones(T_q, T_k, dtype=torch.bool, device=scores.device),
                diagonal=-self.sliding_window,
            )
            causal = causal | far_past
        return scores.masked_fill(causal, float("-inf"))

    def forward(self, x, offset=0):
        B, T, C = x.shape

        q = self.W_q(x).view(B, T, self.n_heads, self.h_dim).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_groups, self.h_dim).transpose(1, 2)

        # Apply RoPE directly — q/k are already (B, H, T, D)
        q = self._compute_rope(q, offset=offset)
        k = self._compute_rope(k, offset=offset)

        # Expand K, V to match Q head count
        k = k.repeat_interleave(self.group_size, dim=1)
        v = v.repeat_interleave(self.group_size, dim=1)

        # QK-Norm (Gemma 3)
        q_norm = F.normalize(q, p=2, dim=-1)
        k_norm = F.normalize(k, p=2, dim=-1)

        scores = torch.matmul(q_norm, k_norm.transpose(-2, -1)) / math.sqrt(self.h_dim)
        scores = self._apply_causal_mask(scores)
        weights = F.softmax(scores, dim=-1)
        out = torch.matmul(weights, v)

        out = out.transpose(1, 2).contiguous().view(B, T, self.n_heads * self.h_dim)
        return self.out_proj(out)


class Gemma3GeGLU(nn.Module):
    """Gemma 3 GeGLU (GELU-based Gated MLP)."""

    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.gate_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.up_proj = nn.Linear(d_in, d_hidden, bias=False)
        self.down_proj = nn.Linear(d_hidden, d_in, bias=False)

    def forward(self, x):
        gate = self.gate_proj(x)
        up = self.up_proj(x)
        return self.down_proj(F.gelu(gate, approximate="tanh") * up)


class Gemma3TransformerBlock(nn.Module):
    """Complete Gemma 3 Transformer block with double-norm."""

    def __init__(
        self,
        dim,
        n_heads,
        n_kv_groups,
        h_dim,
        hidden_dim,
        is_local=True,
        sliding_window=None,
    ):
        super().__init__()
        self.attn = Gemma3GQA(
            dim, n_heads, n_kv_groups, h_dim, is_local, sliding_window
        )
        self.ffn = Gemma3GeGLU(dim, hidden_dim)
        self.input_layernorm = Gemma3RMSNorm(dim)
        self.post_attn_layernorm = Gemma3RMSNorm(dim)
        self.pre_feed_layernorm = Gemma3RMSNorm(dim)
        self.post_feed_layernorm = Gemma3RMSNorm(dim)
        self.is_local = is_local

    def forward(self, x, offset=0):
        # Attention sub-block with double-norm
        shortcut = x
        x = self.input_layernorm(x)
        x = self.attn(x, offset=offset)
        x = self.post_attn_layernorm(x)
        x = shortcut + x

        # MLP sub-block with double-norm
        shortcut = x
        x = self.pre_feed_layernorm(x)
        x = self.ffn(x)
        x = self.post_feed_layernorm(x)
        return shortcut + x

---

## 🧩 Assembling the Complete Gemma 3 Model

Here we assemble our individual components into the complete **Gemma 3 Decoder-Only Transformer Model**.

### 🗺️ Full Architecture Data Flow:

$$\text{Token IDs } [B, T]$$

$$\downarrow$$

$$\text{Embedding Projection } \rightarrow [B, T, 768] \times \sqrt{768} \quad \text{(Embedding Scale)}$$

$$\downarrow$$

$$\text{Transformer Blocks } \times 8 \quad \text{(Alternating Local/Global Blocks with RoPE \& Causal Masks)}$$

$$\downarrow$$

$$\text{Final Layer Norm } \rightarrow \text{RMSNorm}(x)$$

$$\downarrow$$

$$\text{LM Output Head } \rightarrow \text{Linear Projection (Tied with Embeddings)}$$

$$\downarrow$$

$$\text{Logit Cap } \rightarrow 30.0 \times \tanh\left(\frac{\text{Logits}}{30.0}\right)$$

$$\downarrow$$

$$\text{Next-Token Probabilities } [B, T, 256,000]$$

### 🔗 Weight Tying Mechanics:
To save memory, the output language model head shares the exact weight matrix as the token embeddings:

$$\text{lm\_head.weight} = \text{tok\_embeddings.weight}$$

This reduces the parameter count of our model by **196.6M parameters**, while regularizing training by sharing features between input token comprehension and output token generation!

In [3]:
class Gemma3Model(nn.Module):
    """
    Complete Gemma 3 decoder-only model (~270M parameters).

    Architecture:
        input_ids → Embedding (×sqrt(hidden_size)) → TransformerBlocks → RMSNorm → LM Head
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Token embedding
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)

        # 8 transformer layers alternating local/global
        self.layers = nn.ModuleList()
        for i in range(config.num_layers):
            is_local = i % 6 != 5  # layers 0-4 local, layer 5 global, repeat
            self.layers.append(
                Gemma3TransformerBlock(
                    dim=config.hidden_size,
                    n_heads=config.num_heads,
                    n_kv_groups=config.num_kv_heads,
                    h_dim=config.head_dim,
                    hidden_dim=config.intermediate_size,
                    is_local=is_local,
                    sliding_window=config.sliding_window if is_local else None,
                )
            )

        # Final normalization
        self.norm = Gemma3RMSNorm(config.hidden_size)

        # Language model head with weight tying
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.lm_head.weight = self.tok_embeddings.weight  # tied!

    def forward(self, input_ids, offset=0):
        # Embedding + Gemma scaling trick
        x = self.tok_embeddings(input_ids) * self.config.hidden_scale

        for layer in self.layers:
            x = layer(x, offset=offset)

        x = self.norm(x)
        logits = self.lm_head(x)
        # Gemma 3 final logit soft-cap
        return logit_cap * torch.tanh(logits / logit_cap)

---

## 3. Instantiate the Full Model


In [4]:
# Create a config object
class Config:
    def __init__(self):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = head_dim
        self.intermediate_size = intermediate_size
        self.sliding_window = sliding_window
        self.max_position_embeddings = max_position_embeddings
        self.hidden_scale = hidden_scale


config = Config()
model = Gemma3Model(config)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
embed_params = sum(p.numel() for p in model.tok_embeddings.parameters())
lm_head_params = sum(p.numel() for p in model.lm_head.parameters())  # tied!
layer_params = sum(p.numel() for p in model.layers.parameters())
norm_params = sum(p.numel() for p in model.norm.parameters())

print(f"Total parameters: {total_params:,} ({total_params / 1e6:.1f}M)")
print(f"\nBreakdown:")
print(f"  Embedding:    {embed_params:>10,}")
print(f"  LM Head:      {lm_head_params:>10,} (tied with embedding)")
print(f"  Transformer Blocks: {layer_params:>10,}")
print(f"  Final Norm: {norm_params:>10,}")
print(f"\nModel Architecture:")
for i, layer in enumerate(model.layers):
    att_typ = "GLOBAL" if not layer.is_local else f"LOCAL(sw={sliding_window})"
    print(f"  Layer {i:2d}: {att_typ}"[:30].ljust(30))

Total parameters: 246,178,560 (246.2M)

Breakdown:
  Embedding:    196,608,000
  LM Head:      196,608,000 (tied with embedding)
  Transformer Blocks: 49,569,792
  Final Norm:        768

Model Architecture:
  Layer  0: LOCAL(sw=1024)    
  Layer  1: LOCAL(sw=1024)    
  Layer  2: LOCAL(sw=1024)    
  Layer  3: LOCAL(sw=1024)    
  Layer  4: LOCAL(sw=1024)    
  Layer  5: GLOBAL            
  Layer  6: LOCAL(sw=1024)    
  Layer  7: LOCAL(sw=1024)    


---

## 4. Generate Text (Greedy Decoding)


In [5]:
# Generate text with greedy decoding and logit soft-capping
def generate(model, prompt_ids, max_length=30):
    """Greedy decoding with Gemma 3 logit soft-capping."""
    model.eval()
    input_ids = prompt_ids.clone()
    generated = []

    for _ in range(max_length):
        with torch.no_grad():
            logits = model(input_ids)

        # Get the last logit and apply Gemma 3 logit soft-capping
        logit = logits[0, -1, :]
        logit = logit_cap * torch.tanh(logit / logit_cap)  # cap at 30.0

        # Greedy: pick the highest logit
        next_token = logit.argmax(dim=-1).unsqueeze(0)

        # Append
        input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)
        generated.append(next_token.item())

        # Stop at eos token
        if next_token.item() == 1:  # <eos> token
            break

    return input_ids[0]

---

## 5. Run the Complete Model!


In [6]:
_VOCAB = [
    # Special tokens (always first)
    "<pad>",
    "<eos>",
    "<bos>",
    "<unk>",
    # Common words
    "hello",
    "world",
    "what",
    "is",
    "gemma",
    "model",
    "good",
    "work",
    "great",
    "better",
    "fast",
    "efficient",
    "build",
    "scratch",
    "train",
    "predict",
    "neural",
    "network",
    "attention",
    "layer",
    "hidden",
    "size",
    "embed",
    "token",
    "output",
    "sequence",
    "correct",
    "thanks",
    "this",
    "a",
    "text",
    "make",
    "show",
    "learn",
    "forward",
    "pass",
    "embedding",
    "position",
    "decode",
    "generate",
    "input",
    "transform",
    "matrix",
    "compute",
    "loss",
    "optim",
    "grad",
    "step",
    "batch",
    "epoch",
]


class SimpleTokenizer:
    """Toy word-level tokenizer for the workshop"""

    # Fixed special token IDs — match Gemma 3 convention
    PAD_ID = 0
    EOS_ID = 1
    BOS_ID = 2
    UNK_ID = 3

    def __init__(self, vocab_size=vocab_size):
        self.word2id: dict[str, int] = {key: value for value, key in enumerate(_VOCAB)}
        self.id2word: dict[int, str] = {v: k for k, v in self.word2id.items()}
        self.vocab = self.word2id

    def encode(self, text: str, add_bos: bool = False) -> list[int]:
        """Tokenise a string into a list of integer token IDs"""
        ids = [self.BOS_ID] if add_bos else []
        for word in text.lower().split():
            clean = "".join(c for c in word if c.isalnum())
            ids.append(self.word2id.get(clean, self.UNK_ID))
        return ids

    def decode(self, ids: list[int], skip_special: bool = False) -> str:
        """Convert a list of token IDs back to a string"""
        special = {self.PAD_ID, self.EOS_ID, self.BOS_ID, self.UNK_ID}
        tokens = []
        for i in ids:
            if skip_special and i in special:
                continue
            tokens.append(self.id2word.get(i, "<unk>"))
        return " ".join(tokens)

    def __len__(self) -> int:
        """Number of tokens in the vocabulary."""
        return len(self.word2id)

    def __repr__(self) -> str:
        return f"SimpleTokenizer(vocab_size={len(self)})"

In [7]:
tokenizer = SimpleTokenizer()

# Try generating with the full model
prompt = "hello world"
prompt_ids = torch.tensor([tokenizer.encode(prompt)])

print(f"\nPrompt: {prompt}")
print(f"Token IDs: {prompt_ids}")
print(f"\nGenerated tokens:")

generated_ids = generate(model, prompt_ids, max_length=10)
generated_text = tokenizer.decode(generated_ids.tolist())
print(f"{generated_ids}")
print(f"{generated_text}")

# With the random weights, this won't make sense -- but it WILL generate!
print()
print("---")
print("The model CAN generate, even with random weights.")
print("Now we can train it!")


Prompt: hello world
Token IDs: tensor([[4, 5]])

Generated tokens:
tensor([4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5])
hello world world world world world world world world world world world

---
The model CAN generate, even with random weights.
Now we can train it!


---

## Architecture Summary


In [8]:
# Print complete architecture diagram
print("=" * 65)
print("  GEMMA 3 ARCHITECTURE (~270M) -- COMPLETE")
print("=" * 65)
print("  Input Tokens (IDs)")
print("       |")
print("       v")
print("  ┌─────────────┐")
print("  │ Embedding(256K, 768)     │")
print("  │ 	imes sqrt(768) = 27.72     │")
print("  └─────────────┘")
print("       │")
print("       v")
print("  ┌──────────────────────────────┐")
print("  │  Transformer Blocks (	imes8)     │")
print("  │                              │")
print("  │  Blocks 0-4: Local (sw=1024) │")
print("  │  Blocks 5:   Global (full)   │")
print("  │                              │")
print("  │  Each block has:             │")
print("  │  - GQA (8h, 2kv, QK-Norm)  │")
print("  │  - GeGLU (768→2048→768)    │")
print("  │  - Double RMSNorm (Pre+Post) │")
print("  └──────────────────────────────┘")
print("       │")
print("       v")
print("  ┌─────────────┐")
print("  │ RMSNorm     │")
print("  └─────────────┘")
print("       │")
print("       v")
print("  ┌───────────────────┐")
print("  │ LM Head (768→256K) │")
print("  │ (tie with embed)  │")
print("  └───────────────────┘")
print("       │")
print("       v")
print("  Logits (tanh(logits/30)) ⭐ ← Gemma 3 cap!")
print("=" * 65)

  GEMMA 3 ARCHITECTURE (~270M) -- COMPLETE
  Input Tokens (IDs)
       |
       v
  ┌─────────────┐
  │ Embedding(256K, 768)     │
  │ 	imes sqrt(768) = 27.72     │
  └─────────────┘
       │
       v
  ┌──────────────────────────────┐
  │  Transformer Blocks (	imes8)     │
  │                              │
  │  Blocks 0-4: Local (sw=1024) │
  │  Blocks 5:   Global (full)   │
  │                              │
  │  Each block has:             │
  │  - GQA (8h, 2kv, QK-Norm)  │
  │  - GeGLU (768→2048→768)    │
  │  - Double RMSNorm (Pre+Post) │
  └──────────────────────────────┘
       │
       v
  ┌─────────────┐
  │ RMSNorm     │
  └─────────────┘
       │
       v
  ┌───────────────────┐
  │ LM Head (768→256K) │
  │ (tie with embed)  │
  └───────────────────┘
       │
       v
  Logits (tanh(logits/30)) ⭐ ← Gemma 3 cap!


---

## Key Takeaway

Congratulations! You've built a complete Gemma 3 model from scratch:

| Component | Innovation |
|---|---|
| Embedding | Scaled by sqrt(hidden_size) |
| QK-Norm | L2-normalized Q and K (no 50.0 tanh cap on scores) |
| GQA | Shared KV heads (8Q : 2KV) |
| GeGLU | GELU-based gating (not SwiGLU) |
| Double RMSNorm | Pre + post on attention AND MLP |
| Logit Cap | tanh(x/30) on final logits |
| KV Cache | 5:1 local/global reduces memory to <15% |
| RoPE | Dual frequencies: 10K (local), 1M	imes8 (global) |

---
[← 07 The Transformer Block](07_the_transformer_block.ipynb) | [09 Inference and Sampling →](09_inference_and_sampling.ipynb)
